# Working with Spark DataFrames: Transformations & Actions

Welcome to the second notebook in our series! Now that you have initialized your first `SparkSession` and seen basic data printing, it is time to master the engine's core mechanics: **Transformations**, **Actions**, and **Lazy Evaluation**.

---

## 1. Lazy Evaluation: How Spark Thinks

One of Spark's most powerful performance optimizations is **Lazy Evaluation**.

* When you call transformation methods (like `filter()`, `select()`, or `withColumn()`), Spark **does not compute the results immediately**.
* Instead, it builds up a **Directed Acyclic Graph (DAG)**—a logical execution plan tracking the lineage of operations applied to your data.
* Computation only triggers when you invoke an **Action** (like `show()`, `count()`, or `write()`). This allows the Catalyst Optimizer to look at the entire pipeline and optimize execution paths before touching disk or memory.

## 2. Transformations: Narrow vs. Wide

Transformations are operations that take a DataFrame and return a new DataFrame. They are categorized into two types based on how data moves across the cluster network:

### A. Narrow Transformations
* **Definition:** Each partition of the parent DataFrame is used by at most one partition of the child DataFrame.
* **Characteristics:** No data movement across the network (no shuffle required).
* **Examples:** `filter()`, `select()`, `withColumn()`, `drop()`.

### B. Wide Transformations
* **Definition:** Multiple child partitions depend on data from multiple parent partitions.
* **Characteristics:** Requires data to be shuffled across the cluster network. This is computationally expensive and network-bound.
* **Examples:** `groupBy()`, `agg()`, `distinct()`, `join()`.

## 3. Setting up the Environment

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, avg, sum

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("TransformationsAndActions") \
    .getOrCreate()

# Create sample e-commerce dataset
data = [
    (1, "Laptop", "Electronics", 1200.0, 5),
    (2, "Mouse", "Electronics", 25.0, 50),
    (3, "Desk", "Furniture", 300.0, 2),
    (4, "Chair", "Furniture", 150.0, 10),
    (5, "Monitor", "Electronics", 350.0, 8),
    (6, "Notebook", "Stationery", 5.0, 100)
]

columns = ["ProductID", "ProductName", "Category", "Price", "Stock"]
df = spark.createDataFrame(data, columns)

# Action: Display initial data
df.show()

## 4. Applying Narrow Transformations

Let's apply some narrow transformations. Notice that no network shuffle occurs here because each partition processes its rows independently.

In [ ]:
# 1. Select specific columns and rename/add a calculated column (Total Inventory Value)
valued_df = df.select(
    col("ProductName"),
    col("Category"),
    (col("Price") * col("Stock")).alias("TotalValue")
)

# 2. Filter rows (Narrow transformation)
expensive_inventory = valued_df.filter(col("TotalValue") > 1000)

# Action triggered here to view results
expensive_inventory.show()

## 5. Applying Wide Transformations (The Shuffle)

Now let's look at a wide transformation. Grouping data requires collecting all rows belonging to the same category from across different worker nodes into a single partition, triggering a **shuffle**.

In [ ]:
# Wide Transformation: Group by category and compute aggregations
category_summary = df.groupBy("Category") \
    .agg(
        avg("Price").alias("Avg_Price"),
        sum("Stock").alias("Total_Stock")
    )

# Action: Show aggregated metrics
category_summary.show()

## Summary of Actions vs. Transformations

* **Transformations (Lazy):** Build the plan (`select`, `filter`, `withColumn`, `groupBy`).
* **Actions (Eager):** Execute the plan and return data (`show()`, `count()`, `collect()`, `take(n)`, `write()`).

Always chain as many transformations as needed and trigger actions only when necessary to keep your Spark jobs performing at maximum speed!